In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"

print("Plotly version:", __import__("plotly").__version__)

In [2]:
import pandas as pd

df = pd.read_csv("Renewable Energy World Wide-energy-data.csv")
print("Shape:", df.shape)
display(df.head())

Shape: (23377, 130)


,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
0,ASEAN (Ember),2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
1,ASEAN (Ember),2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
2,ASEAN (Ember),2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
3,ASEAN (Ember),2003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
4,ASEAN (Ember),2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN


In [6]:
import plotly.express as px

# Let's visualize the distribution of renewable electricity generation share across all entries
# We will use 'renewables_share_elec' since it represents a core metric in this dataset.
if 'renewables_share_elec' in df.columns:
    fig1 = px.histogram(df.dropna(subset=['renewables_share_elec']),
                        x="renewables_share_elec",
                        title="Distribution of Renewable Share in Electricity Generation (%)",
                        labels={"renewables_share_elec": "Renewable Share (%)"})
    fig1.update_layout(yaxis_title="Number of records", bargap=0.05)
    fig1.show()
else:
    print("Column 'renewables_share_elec' not found in dataset. Checking available columns...")
    print([col for col in df.columns if 'share' in col or 'renewables' in col][:10])

In [7]:
import pandas as pd
import plotly.express as px

# Filter for some major countries and plot the growth of wind and solar consumption over the years
countries_to_plot = ["United States", "China", "India", "Germany", "Brazil", "United Kingdom"]
plot_df = df[df['country'].isin(countries_to_plot)].copy()

# Ensure year is treated correctly
plot_df['year'] = pd.to_numeric(plot_df['year'], errors='coerce')
plot_df = plot_df.dropna(subset=['year', 'renewables_consumption']).copy()

fig2 = px.line(
    plot_df,
    x="year",
    y="renewables_consumption",
    color="country",
    title="Renewable Energy Consumption Over Time by Country (TWh)",
    labels={
        "year": "Year",
        "renewables_consumption": "Renewables Consumption (TWh)",
        "country": "Country"
    }
)
fig2.show()

In [8]:
# Compare GDP vs Renewable Energy Consumption in a scatter plot
import plotly.express as px

# Filter to valid positive values for log scaling
scatter_df = df.dropna(subset=['gdp', 'renewables_consumption', 'country']).copy()
scatter_df = scatter_df[(scatter_df['gdp'] > 0) & (scatter_df['renewables_consumption'] > 0)]

# Get the most recent data year per country
idx = scatter_df.groupby('country')['year'].idxmax()
latest_scatter_df = scatter_df.loc[idx]

fig3 = px.scatter(
    latest_scatter_df,
    x="gdp",
    y="renewables_consumption",
    hover_name="country",
    size="population" if "population" in latest_scatter_df.columns else None,
    log_x=True,
    log_y=True,
    title="GDP vs. Renewable Energy Consumption (Latest Year Available)",
    labels={
        "gdp": "GDP (log scale)",
        "renewables_consumption": "Renewables Consumption (TWh, log scale)"
    }
)
fig3.show()

In [10]:
import pandas as pd
import plotly.express as px

# Let's compare the share of electricity generated from different renewable sources (Wind vs Solar) for recent years.
# We will use 'wind_share_elec' and 'solar_share_elec'.
plot_df = df.dropna(subset=['wind_share_elec', 'solar_share_elec', 'gdp', 'country']).copy()
# Limit to the most recent year available per country
idx = plot_df.groupby('country')['year'].idxmax()
latest_df = plot_df.loc[idx]

fig2 = px.scatter(
    latest_df,
    x="wind_share_elec",
    y="solar_share_elec",
    hover_name="country",
    size="population" if "population" in latest_df.columns else None,
    color="country",
    title="Wind Share vs. Solar Share in Electricity Generation (Latest Year)",
    labels={
        "wind_share_elec": "Wind Share of Electricity (%)",
        "solar_share_elec": "Solar Share of Electricity (%)"
    }
)
fig2.show()

In [11]:
import pandas as pd
import plotly.express as px

# Let's visualize the distribution of low-carbon electricity share globally.
# We can plot a map if we have ISO codes, or a sorted bar chart of the top 30 countries.
plot_df = df.dropna(subset=['low_carbon_share_elec', 'country', 'year']).copy()
idx = plot_df.groupby('country')['year'].idxmax()
latest_df = plot_df.loc[idx]

# Exclude region aggregates (which usually do not have standard iso_codes)
top_countries = latest_df.dropna(subset=['iso_code']).sort_values(by='low_carbon_share_elec', ascending=False).head(30)

fig3 = px.bar(
    top_countries,
    x="low_carbon_share_elec",
    y="country",
    orientation='h',
    title="Top 30 Countries by Low-Carbon Share in Electricity Generation (Latest Year)",
    labels={"low_carbon_share_elec": "Low-Carbon Electricity Share (%)", "country": "Country"},
    color="low_carbon_share_elec",
    color_continuous_scale="Viridis"
)
fig3.update_layout(yaxis={'categoryorder':'total ascending'})
fig3.show()

In [12]:
import pandas as pd
import plotly.express as px

# Let's add a new visualization showing the global evolution of the renewable mix (Hydro, Wind, Solar)
# We'll filter for 'World' aggregate over time
world_df = df[df['country'] == 'World'].copy()
world_df['year'] = pd.to_numeric(world_df['year'], errors='coerce')
world_df = world_df[world_df['year'] >= 1990].copy()

# Prepare data for a stacked area chart
melted_df = world_df.melt(
    id_vars=['year'],
    value_vars=['hydro_electricity', 'wind_electricity', 'solar_electricity'],
    var_name='Energy Source',
    value_name='Generation (TWh)'
)

# Clean names for presentation
melted_df['Energy Source'] = melted_df['Energy Source'].str.replace('_electricity', '').str.capitalize()

fig_extra = px.area(
    melted_df,
    x="year",
    y="Generation (TWh)",
    color="Energy Source",
    title="Global Renewable Electricity Generation by Source (1990-Present)",
    labels={"year": "Year", "Generation (TWh)": "Electricity Generation (TWh)"},
    color_discrete_sequence=px.colors.qualitative.Safe
)
fig_extra.show()